# Updated Sample Modeling Notebook

Comprehensive from-scratch tree/ensemble modeling + hypothesis testing on teammate sample dataset.


In [1]:
from pathlib import Path
import sys

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from updated_sample.pipeline import load_data, feature_engineering, temporal_split, make_models, eval_metrics, pairwise_stat_tests, hypothesis_tests


In [2]:
csv_path = ROOT / "data" / "raw" / "yellow_taxi_representative_sample_2021_2023_distribution.csv"
df = load_data(csv_path)
feat = feature_engineering(df)
split = temporal_split(feat)
print(df.shape, feat.shape, len(split.train), len(split.valid), len(split.test))


(36, 4) (33, 14) 23 5 5


In [3]:
features = ["year", "month", "quarter", "month_sin", "month_cos", "lag_1", "lag_2", "lag_3", "rolling_mean_3", "rolling_std_3"]
target = "sample_rows"
models = make_models(seed=42)

import pandas as pd
rows=[]
preds = pd.DataFrame(index=split.test.index)
for name, model in models.items():
    model.fit(split.train[features], split.train[target])
    p = model.predict(split.test[features])
    preds[name] = p
    m = eval_metrics(split.test[target].to_numpy(), p)
    m["model"] = name
    rows.append(m)
model_df = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
model_df


,mae,rmse,r2,residual_variance,model
0,1.212091,1.275717,-9.171595,0.158290,boosting_xgboost
1,2.689041,2.715370,-45.082722,0.142296,stacking_ensemble
2,3.116000,3.139414,-60.599500,0.146464,bagging_random_forest


In [4]:
pairwise_df = pairwise_stat_tests(split.test[target].to_numpy(), preds, n_bootstrap=3000, n_perm=5000, seed=42)
pairwise_df


,model_a,model_b,mean_abs_error_diff_a_minus_b,ci95_low,ci95_high,p_permutation,p_wilcoxon
0,boosting_xgboost,bagging_random_forest,-1.903909,-1.925486,-1.882332,0.061988,0.0625
1,boosting_xgboost,stacking_ensemble,-1.476949,-1.883438,-0.701621,0.123975,0.1250
2,bagging_random_forest,stacking_ensemble,0.426959,0.010221,1.191429,0.123175,0.1250


In [5]:
hyp = hypothesis_tests(feat, model_df, pairwise_df, alpha=0.05)
hyp


{'h1_seasonality': {'tested': True,
  'p_value': 0.9520819621960708,
  'support': False,
  'description': 'Monthly seasonality effect exists in sample_rows.'},
 'h2_ensemble_better_than_boosting': {'tested': True,
  'support': False,
  'rule': 'At least one ensemble beats boosting on MAE and shows significant paired error difference.',
  'boosting_mae': 1.212091088294983,
  'bagging_mae': 3.115999999999997,
  'stacking_mae': 2.689040556464363,
  'pairwise_details': [{'pair': 'boosting_xgboost vs bagging_random_forest',
    'p_permutation': 0.0619876024795041,
    'ci95': [-1.9254858398437478, -1.8823320312499958],
    'direction_support': False,
    'passed': False},
   {'pair': 'boosting_xgboost vs stacking_ensemble',
    'p_permutation': 0.1239752049590082,
    'ci95': [-1.8834384639044686, -0.7016211287617693],
    'direction_support': False,
    'passed': False}]}}

In [6]:
from updated_sample.pipeline import make_plots
out_fig = ROOT / "reports" / "figures"
out_fig.mkdir(parents=True, exist_ok=True)
make_plots(model_df, preds, split.test[target].to_numpy(), out_fig)
print("saved:", out_fig)


saved: /Users/heliamahmoodzadeh/Documents/GitHub/nyc-mobility-analysis/updated_GITHUB_sample_code/reports/figures
